# 📊 Strategic Marketing Analytics & Capital Reallocation Engine

**Objective:** To identify capital-at-risk within a global digital advertising portfolio ($54M Revenue / 4.88X ROAS) and algorithmically reallocate wasted spend to high-performing assets using strict financial guardrails.

**Data Context:** This audit analyzes 1,800 global operational days extracted from the `global_ad_ops_warehouse` SQLite database. The dataset encompasses cross-platform daily performance across **Google Ads, Meta Ads, and TikTok Ads** to identify hidden daily inefficiencies that get lost in macro-level monthly reporting.

---

## 1. Executive Summary
In performance marketing, macro-level reporting often hides micro-level bleeding. A campaign might look profitable on a monthly basis, but lose thousands of dollars on specific conditions. This project focuses on **Surgical Capital Reallocation** to transition the enterprise from a 4.88X baseline to a **5.21X ROAS frontier** while strictly decreasing total capital exposure.

**💰 Financial Baseline (2024):**
* **Total Revenue:** $54.0M
* **Global ROAS:** 4.88X

**🚀 The Algorithmic Solution & Impact:**
We built a micro-simulation engine that identifies specific days yielding unacceptable CPAs (The "Death Zone") and reroutes that capital. Applying strict industry guardrails (a 20% daily budget scale cap and an 80% marginal efficiency penalty), the engine achieves the following:

* **Capital Extracted:** Identified **\$820,366**, 7.38% of the total ad spend of inefficient capital actively bleeding profit.
* **Safe Reallocation:** Surgically reinvested **\$357,101** into high-velocity "Star Segments" operating on the exact same day.
* **Pure Cash Savings:** Withheld **\$463,264** as unspent capital (pure profit saved) to protect platform algorithms from over-scaling.
* **Profit Optimization:** Drove **\$1.2M** in total profit uplift (combining \$583K in withheld cash savings and \$617K in net-new revenue) through automated capital reallocation.

**Conclusion:** This mathematical efficiency gain drives the global enterprise ROAS from a baseline of **4.88X** to **5.21X**.



In [ ]:
# ==========================================
# 2. DATA INGESTION & INTEGRITY AUDIT
# ==========================================
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore') # Keeps notebook clean
sns.set_theme(style="whitegrid")  # Gives charts a professional portfolio look
# ------------------------------------------------

db_path = 'global_ad_ops_warehouse.db'

# --- 1. DEFENSIVE DATABASE INGESTION ---
if not os.path.exists(db_path):
    raise FileNotFoundError(f"⚠️ SYSTEM ERROR: Database file '{db_path}' not found.")

try:
    conn = sqlite3.connect(db_path)
    df = pd.read_sql_query("SELECT * FROM vw_performance_metrics", conn)
except Exception as e:
    raise RuntimeError(f"⚠️ DATABASE ERROR: {e}")
finally:
    conn.close()

# Pre-processing
df['date'] = pd.to_datetime(df['date'])
df['is_optimization_target'] = df['net_profit'] < 0

# --- 2. ENHANCED DATA QUALITY & INTEGRITY AUDIT ---
def check_data_quality(dataframe: pd.DataFrame) -> None:
    """
    Performs a clinical audit of the dataset to identify structural risks.
    Distinguishes between technical errors and business-logic overlaps.
    """
    total_records = len(dataframe)
    null_values = dataframe.isnull().sum().sum()

    # A. Technical Duplicate Check (Ignoring DB Primary Key)
    cols_to_check = [col for col in dataframe.columns if col != 'row_id']
    exact_duplicates = dataframe[cols_to_check].duplicated().sum()

    # B. Logical Overlap Check (Grain Analysis)
    logical_keys = ['date', 'platform', 'campaign_type', 'industry', 'country']
    logical_overlaps = dataframe.duplicated(subset=logical_keys).sum()

    # C. Continuity Audit (Date Gaps)
    expected_dates = pd.date_range(start=dataframe['date'].min(), end=dataframe['date'].max())
    missing_dates = len(expected_dates.difference(dataframe['date']))

    # D. Funnel Integrity Check
    logic_errors = len(dataframe[dataframe['clicks'] > dataframe['impressions']]) + \
                   len(dataframe[dataframe['conversions'] > dataframe['clicks']])

    # Executive Output
    print("=========================================================")
    print(" 🛡️ SYSTEM: DATA INTEGRITY & QUALITY REPORT")
    print("=========================================================")
    print(f"Total Records Ingested:          {total_records:,}")
    print(f"Missing/Null Values:             {null_values}")
    print(f"Technical Duplicates:            {exact_duplicates}")
    print(f"Logical Overlaps (Same Grain):   {logical_overlaps}")
    print(f"Missing Operational Days:        {missing_dates}")
    print(f"Funnel Logic Errors:             {logic_errors}")
    print("=========================================================\n")

    if exact_duplicates == 0 and logic_errors == 0:
        print("✅ STATUS: Technical integrity verified.")
        if logical_overlaps > 0 or missing_dates > 0:
            print("⚠️ NOTE: Structural anomalies detected. See 'Structural Note' for assessment.")
    else:
        print("🛑 WARNING: Data cleaning required.")

# Execute the audit
check_data_quality(df)

### 🔍 Structural Note: Data Grain & Pipeline Audit
Upon execution of the data integrity audit, two primary structural anomalies were identified: **12 configuration overlaps** and **5 missing operational dates**.

**1. Assessment of Overlaps (The "Grain" Analysis)**
* **Observation:** While the logical key (Date/Platform/Type/Industry/Country) repeats, a deep-dive into the raw records confirms that the financial metrics (*Ad Spend* and *Revenue*) vary between these entries.
* **Root Cause:** This suggests the database is capturing **distinct sub-campaigns or secondary ad accounts** within the same geographic grain (e.g., separate 'Mens' and 'Womens' shopping accounts for the UK) rather than technical row duplication.
* **Strategic Action:** To ensure 100% of the $54M ecosystem revenue is audited and accounted for, these rows are treated as **additive**. Removing them would artificially deflate the global baseline.

**2. Assessment of Missing Dates (The "Continuity" Analysis)**
* **Observation:** The audit flagged 5 dates missing from the 2024 fiscal year (e.g., April 23, July 15).
* **Root Cause:** These gaps typically represent upstream API sync timeouts or server maintenance during the **ETL (Extract, Transform, Load)** process from platform APIs (Google/Meta/TikTok) to the SQLite warehouse.
* **Strategic Action:** Representing less than **1.5%** of the total temporal dataset, this gap is considered **statistically immaterial**. The global 4.88X ROAS baseline remains representative, and the analysis proceeds using the existing temporal distribution.

> **✅ Conclusion:** The dataset is technically sound, structurally accounted for, and cleared for the Strategic Sensitivity Analysis.

In [ ]:
# ==========================================
# 2. PLATFORM-LEVEL PORTFOLIO OVERVIEW
# ==========================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
from IPython.display import display

# Set a clean, modern style
sns.set_theme(style="white", font="sans-serif")
plt.rcParams['text.color'] = '#333333'
plt.rcParams['axes.labelcolor'] = '#333333'
plt.rcParams['xtick.color'] = '#333333'
plt.rcParams['ytick.color'] = '#333333'

# Engineer the profit metric before aggregating
if 'profit' not in df.columns:
    df['profit'] = df['revenue'] - df['ad_spend']

# -------------------------------
# 1️⃣ Aggregate platform metrics
# -------------------------------
platform_summary = (
    df.groupby("platform")
      .agg(
          total_spend=("ad_spend", "sum"),
          total_revenue=("revenue", "sum"),
          total_profit=("profit", "sum"),
          total_conversions=("conversions", "sum")
      )
      .reset_index()
)

# Calculate shares dynamically
for col in ["spend", "revenue", "profit"]:
    platform_summary[f"{col}_share"] = platform_summary[f"total_{col}"] / platform_summary[f"total_{col}"].sum()

# Sort by spend to keep visual hierarchy consistent
platform_summary = platform_summary.sort_values("total_spend", ascending=False)

# Display clean table
display(platform_summary.style.format({
    'total_spend': '${:,.0f}',
    'total_revenue': '${:,.0f}',
    'total_profit': '${:,.0f}',
    'spend_share': '{:.1%}',
    'revenue_share': '{:.1%}',
    'profit_share': '{:.1%}'
}).hide(axis="index"))

# -------------------------------
# 2️⃣ Define platform colors (High Contrast)
# -------------------------------
platform_colors = {
    "Google Ads": "#1A73E8",   # Corporate Google Blue
    "Meta Ads": "#00BFA5",     # High-contrast Teal
    "TikTok Ads": "#202124"    # Charcoal Black
}
colors = platform_summary["platform"].map(platform_colors).tolist()

# -------------------------------
# 3️⃣ Master Dashboard Visualization
# -------------------------------
# Create a custom layout with 2 rows. Top row has 2 columns, bottom row spans both.
fig = plt.figure(figsize=(14, 12))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.2], hspace=0.3)

# Main overarching title
fig.suptitle('Capital Misallocation Dashboard: Spend Concentration vs. Profit Generation',
             fontsize=20, fontweight='bold', y=0.98)

# ==========================================
# Chart 1 (Top Left): Donut Chart (Spend)
# ==========================================
ax1 = fig.add_subplot(gs[0, 0])
wedges, texts, autotexts = ax1.pie(
    platform_summary["spend_share"],
    labels=platform_summary["platform"],
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops={'width': 0.4, 'edgecolor': 'white', 'linewidth': 3}
)
plt.setp(autotexts, size=11, weight="bold", color="white")
plt.setp(texts, size=12)

total_spend_m = platform_summary["total_spend"].sum() / 1_000_000
ax1.text(0, 0, f"Total Spend\n${total_spend_m:.1f}M", ha='center', va='center', fontsize=14, fontweight='bold')
ax1.set_title("Where the Budget Goes\n(Spend Share)", fontsize=14, pad=15)

# ==========================================
# Chart 2 (Top Right): Single Bar Chart (Profit)
# ==========================================
ax2 = fig.add_subplot(gs[0, 1])
bars2 = ax2.bar(platform_summary["platform"], platform_summary["profit_share"] * 100, color=colors, width=0.6)
ax2.set_title("Where the Returns Come From\n(Profit Share)", fontsize=14, pad=15)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.tick_params(left=False)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.bar_label(bars2, fmt='%.1f%%', padding=5, fontsize=12, fontweight='bold')
ax2.grid(axis='y', linestyle='--', alpha=0.4)
ax2.set_axisbelow(True)

# ==========================================
# Chart 3 (Bottom): Grouped Bar Chart Comparison
# ==========================================
ax3 = fig.add_subplot(gs[1, :]) # Spans all columns in row 1
x = np.arange(len(platform_summary["platform"]))
width = 0.35

bars_spend = ax3.bar(x - width/2, platform_summary["spend_share"] * 100, width, label='Spend Share', color='#1A73E8')
bars_profit = ax3.bar(x + width/2, platform_summary["profit_share"] * 100, width, label='Profit Share', color='#00BFA5')

ax3.set_title('The Delta: Spend Share vs. Profit Share by Platform', fontsize=16, fontweight='bold', pad=15)
ax3.set_xticks(x)
ax3.set_xticklabels(platform_summary["platform"], fontsize=13, fontweight='bold')
ax3.legend(loc='upper right', frameon=False, fontsize=12)

ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
ax3.spines['left'].set_visible(False)
ax3.tick_params(left=False)
ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
ax3.bar_label(bars_spend, fmt='%.1f%%', padding=4, fontsize=12, color='#1A73E8', fontweight='bold')
ax3.bar_label(bars_profit, fmt='%.1f%%', padding=4, fontsize=12, color='#00BFA5', fontweight='bold')
ax3.grid(axis='y', linestyle='--', alpha=0.4)
ax3.set_axisbelow(True)

# Added tight_layout with a rect parameter to prevent the suptitle from overlapping the subplots
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 4. Strategic Profitability Analysis: Uncovering True Business Drivers

While gross revenue indicates scale, **profitability reflects structural efficiency**. This section moves beyond surface-level volume metrics to mathematically isolate which strategic levers (Platform, Country, Campaign Type, and Industry) actually drive bottom-line margin.

To ensure our insights are statistically sound and free from confounding variables, we deploy a strict econometric approach rather than relying on basic descriptive aggregates:

* **Statistical Variance Isolation (Controlled ANOVA):** Because profit is mathematically derived from budget size ($Profit = Revenue - Spend$), basic predictive models or aggregate charts will naturally assign massive weight simply to `ad_spend`. To uncover the *structural* efficiencies of our campaigns, we fit an Ordinary Least Squares (OLS) regression and perform an Analysis of Variance (ANOVA).
* **Controlling for Scale:** Crucially, by holding `ad_spend` constant within the ANOVA model, we strip away the noise of budget volume. This allows us to calculate the independent, statistically significant ($p < 0.05$) impact of categorical elements—like Platform and Country—on portfolio profitability.

**Business Objective:** This econometric approach prevents the classic marketing trap of mistaking "biggest spenders" for "best performers." By identifying which specific platforms inherently convert capital into profit most efficiently with statistical certainty, we can confidently dictate data-driven capital reallocation mandates.

In [ ]:
# =========================================================
# 3. STRATEGIC PROFITABILITY ANALYSIS (ANOVA)
# Isolating the true structural drivers of efficiency
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams['text.color'] = '#333333'

print("=========================================================")
print(" STRATEGIC PROFITABILITY ANALYSIS")
print("=========================================================\n")

# ---------------------------------------------------------
# 1️⃣ CREATE PROFIT METRIC
# ---------------------------------------------------------
print("1️⃣ Establishing the Profit Baseline\n")

# Calculate Profit
df['profit'] = df['revenue'] - df['ad_spend']

# Display formatted summary
profit_summary = df['profit'].describe().to_frame().T
display(profit_summary.style.format("{:,.2f}").hide(axis="index"))

# ---------------------------------------------------------
# 2️⃣ CONTROLLED ANOVA (STATISTICAL SIGNIFICANCE)
# ---------------------------------------------------------
print("\n2️⃣ Statistical Impact on Profit (Controlling for Spend)\n")

# Objective: We know 'ad_spend' drives the sheer volume of profit.
# But which structural categories (Platform, Country, etc.) drive *efficiency*?
# We use OLS + ANOVA to hold ad_spend constant and test the categories.

model = smf.ols(
    'profit ~ ad_spend + C(platform) + C(country) + C(campaign_type) + C(industry)',
    data=df
).fit()

anova_table = sm.stats.anova_lm(model, typ=2)
anova_clean = anova_table.reset_index()
anova_clean.columns = ['Variable', 'Sum of Squares', 'DF', 'F-Score', 'P-Value']

# Filter out the Residuals and clean variable names
anova_clean = anova_clean[anova_clean['Variable'] != 'Residual'].copy()
anova_clean['Variable'] = anova_clean['Variable'].str.replace(r'C\((.*?)\)', r'\1', regex=True)

# ---- Styled Statistical Table ----
def highlight_significance(row):
    # Highlight row in green if statistically significant (p < 0.05)
    if row['P-Value'] < 0.05:
        return ['color: #1b7837; font-weight: bold'] * len(row)
    else:
        return ['color: #757575'] * len(row)

display(
    anova_clean[['Variable', 'F-Score', 'P-Value']]
    .sort_values(by='F-Score', ascending=False)
    .style
    .format({'F-Score': '{:.2f}', 'P-Value': '{:.5f}'})
    .apply(highlight_significance, axis=1)
    .hide(axis="index")
)

# ---- Executive Visualization ----
# Exclude ad_spend to focus purely on the categorical drivers
anova_plot_data = anova_clean[anova_clean['Variable'] != 'ad_spend'].sort_values("F-Score", ascending=False)

plt.figure(figsize=(9, 4))

# Color code: Blue for significant, Grey for noise
colors = ["#1A73E8" if p < 0.05 else "#E0E0E0" for p in anova_plot_data["P-Value"]]

ax = sns.barplot(
    data=anova_plot_data,
    x="F-Score",
    y="Variable",
    palette=colors
)

plt.title("Isolating Structural Profit Drivers (ANOVA)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("F-Statistic (Impact Strength)", fontweight='bold')
plt.ylabel("")

# Add data labels
for i, v in enumerate(anova_plot_data["F-Score"]):
    font_weight = 'bold' if anova_plot_data.iloc[i]["P-Value"] < 0.05 else 'normal'
    ax.text(v + (v*0.01), i, f" {v:.1f}", va='center', fontweight=font_weight)

# Add a legend/footnote
plt.figtext(0.95, -0.05, "Blue indicates Statistical Significance (p < 0.05)",
            ha="right", fontsize=10, style='italic', color="#1A73E8")

sns.despine()
plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 3️⃣ EXECUTIVE INSIGHTS
# ---------------------------------------------------------
print("\n=========================================================")
print(" KEY PROFITABILITY INSIGHTS")
print("=========================================================\n")

top_anova = anova_plot_data.iloc[0]['Variable']
p_val = anova_plot_data.iloc[0]['P-Value']

print(f"📊 MOST SIGNIFICANT STRUCTURAL DRIVER: {top_anova.upper()}")
print(f"   • F-Score: {anova_plot_data.iloc[0]['F-Score']:.1f}")
print(f"   • P-Value: {p_val:.5f}")

print("\n💡 STRATEGIC TAKEAWAY:")
print(f"While total ad spend predictably dictates the sheer volume of profit, the ANOVA test")
print(f"proves that '{top_anova}' is the only structural variable that significantly alters")
print(f"profit efficiency (p < 0.05). Variance in Country, Industry, and Campaign Type")
print(f"are statistically insignificant and should not dictate high-level budget allocation.")

## 5. Identifying "Star Segments" (Capital Receivers)
Before we extract money from failing campaigns, we must prove that we have highly profitable environments ready to absorb that capital. Below are the top 20 most profitable segments in the ecosystem—these will act as our primary "Receivers" during the algorithmic reallocation.

In [ ]:
# ==========================================
# 5. STRATEGIC TARGETING: THE "STAR SEGMENTS"
# ==========================================
from IPython.display import display

print("=========================================================")
print(" ⭐ PROOF OF LIQUIDITY: TOP 20 'STAR SEGMENTS'")
print("=========================================================")
print("Objective: Identify high-efficiency environments with proven volume to safely absorb capital.\n")

# 1. Define the strategic columns for executive review
cols_to_show = ['date', 'platform', 'campaign_type', 'country', 'ad_spend', 'revenue', 'net_profit', 'roas']

# 2. THE BUSINESS LOGIC: Volume-Filtered Efficiency
# Guardrail: Only select campaigns that have spent above the 50th percentile (Median).
# This guarantees they have a massive, proven audience pool ready to absorb the newly injected capital.
median_spend = df['ad_spend'].median()
volume_stars = df[df['ad_spend'] >= median_spend]

# Sort those high-volume, proven campaigns by the highest ROAS
top_20_stars = volume_stars.sort_values(by='roas', ascending=False).head(20)

# 3. Apply financial formatting and visual gradients
styled_top_20 = top_20_stars[cols_to_show].style.format({
    'ad_spend': '${:,.2f}',
    'revenue': '${:,.2f}',
    'net_profit': '${:,.2f}',
    'roas': '{:.2f}X'
}).background_gradient(subset=['roas'], cmap='Greens')

# Display the final styled dataframe
display(styled_top_20)

## 6. Capital Bleed Analysis: The "Death Zone" Audit
To fuel our Reallocation Engine, we must first extract liquidity from failing environments. Below, we isolate the specific campaign days that yielded a negative net profit. By automatically identifying these "Death Zone" segments, we can trigger an algorithmic pause and rescue the remaining daily budget.

In [ ]:
# --- LOSING DAYS AUDIT LOG (136 DAYS) ---



# 1. Define the columns to display

cols_to_show = ['date', 'country', 'platform', 'industry', 'ad_spend', 'revenue', 'net_profit', 'roas']



# 2. Extract the 136 losing days, sorted by severity (worst losses at the top)

losing_days = df[df['net_profit'] < 0].sort_values(by='net_profit', ascending=True)



print(f"--- LOSING DAYS AUDIT LOG ({len(losing_days)} DAYS IDENTIFIED) ---")



# 3. Apply styling with the correct syntax [brackets]

styled_losing_days = losing_days[cols_to_show].style.format({

    'ad_spend': '${:,.2f}',

    'revenue': '${:,.2f}',

    'net_profit': '${:,.2f}',

    'roas': '{:.2f}X'

}).background_gradient(subset=['net_profit'], cmap='Reds_r')



# 4. Render the table (Must be the very last line)

styled_losing_days

## 7.Root Cause Analysis: Uncovering Strategic Inefficiencies
While the global operation maintains a healthy 4.88X ROAS baseline, a **clinical audit of the 136 underperforming days** reveals specific operational friction points.

This section transitions from identification to **Diagnostic Analysis**. We will break down the "Why" behind the capital-at-risk by analyzing the variance between our high-performing baselines and these specific underperforming anomalies.

**Audit Objectives:**
1. **Platform Variance:** Identifying which platforms are most susceptible to high-CPA "Death Zone" events.
2. **Correlation Audit:** Determining if specific campaign types or geographic regions are statistically over-represented in the loss column.
3. **Liquidity Recovery:** Establishing the exact dollar amount available for extraction and reallocation.

In [ ]:
# ==============================================================================
# 7. ROOT CAUSE ANALYSIS: THE SEARCH FOR THE "KILL-SWITCH"
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("=========================================================")
print(" 🔍 DIAGNOSTIC AUDIT: IDENTIFYING THE 'KILL-SWITCH'")
print("=========================================================\n")

# --- 1. Technical Variance Profile ---
# We compare Profitable vs. Losing segments to see which metric 'breaks' first.
metrics_to_audit = ['ctr', 'cpc', 'ad_spend', 'cpa', 'roas']
variance_profile = df.groupby('is_optimization_target')[metrics_to_audit].mean().T
variance_profile.columns = ['Profitable_Baseline', 'Losing_Anomalies']

# Mathematical Variance Calculation
variance_profile['Variance_Abs'] = variance_profile['Losing_Anomalies'] - variance_profile['Profitable_Baseline']
variance_profile['Variance_Pct'] = (variance_profile['Variance_Abs'] / variance_profile['Profitable_Baseline']) * 100

print("--- 1. METRIC VARIANCE: BASELINE VS. ANOMALY ---")
display(variance_profile.style.format({
    'Profitable_Baseline': '{:.2f}',
    'Losing_Anomalies': '{:.2f}',
    'Variance_Abs': '{:+.2f}',
    'Variance_Pct': '{:+.2f}%'
}).background_gradient(subset=['Variance_Pct'], cmap='RdYlGn_r'))

# --- 2. Stress-Testing CPA Thresholds (The Kill-Switch) ---
# Goal: Find the 'Point of No Return' where loss probability exceeds 90%.
print("\n--- 2. PROBABILITY AUDIT: CPA THRESHOLD SENSITIVITY ---")
threshold_results = []

# NEW ADDITION: Calculate the low-risk Star Segment first (<$100)
star_subset = df[(df['cpa'] < 100) & (df['cpa'] > 0)] # >0 filters out zero-spend days
if not star_subset.empty:
    star_loss_prob = (len(star_subset[star_subset['net_profit'] < 0]) / len(star_subset))
    threshold_results.append({
        "CPA_Threshold": "<$100 (Star)",
        "Loss_Probability": star_loss_prob,
        "Sample_Size": len(star_subset),
        "Recommendation": "SCALE/INJECT" # Positive directive for the BI dashboard
    })

# ORIGINAL LOOP: Calculate the high-risk tiers
for threshold in [100, 150, 200, 250]:
    subset = df[df['cpa'] > threshold]
    if not subset.empty:
        loss_prob = (len(subset[subset['net_profit'] < 0]) / len(subset))
        threshold_results.append({
            "CPA_Threshold": f">${threshold}+",
            "Loss_Probability": loss_prob,
            "Sample_Size": len(subset),
            "Recommendation": "PAUSE" if loss_prob > 0.8 else "MONITOR"
        })

threshold_df = pd.DataFrame(threshold_results)
# Use a custom colormap in pandas to show green for low risk, red for high risk
display(threshold_df.style.format({
    'Loss_Probability': '{:.1%}'
}).background_gradient(subset=['Loss_Probability'], cmap='RdYlGn_r'))

# --- 3. Risk Correlation Visual (The 'Bleeding' Map) ---
# Replacing Plotly with a professional Seaborn FacetGrid for consistency.
plt.figure(figsize=(12, 6))
g = sns.FacetGrid(df[df['is_optimization_target']], col="platform", hue="platform", palette="Set1", height=4)
g.map(sns.scatterplot, "ad_spend", "cpa", alpha=0.6)
g.add_legend()

g.fig.subplots_adjust(top=0.8)
g.fig.suptitle('Risk Correlation: Ad Spend vs. CPA Inflation (Losing Days Only)', fontsize=14, fontweight='bold')

for ax in g.axes.flat:
    ax.axhline(250, ls='--', color='red', alpha=0.5, label='Death Zone') # The $250 Kill-Switch

plt.show()

print("\n🚨 DIAGNOSTIC CONCLUSION:")
print("The data confirms a 'Critical Failure Point' at the $250 CPA threshold.")
print("Campaigns exceeding this limit have a >95% probability of net-loss.")
print("--> STRATEGIC FIX: Implement an automated hard-stop at $250 CPA to rescue liquidity.")

### 🔍 Root Cause Insight: The "Seasonality Trap"
The variance profile above reveals a critical operational flaw: on losing days, **CPC inflates by ~56%** and **Ad Spend spikes by ~51%**, resulting in a devastating **184% increase in CPA**.

**The Business Context:**
This data signature is a classic symptom of the **Seasonality Trap**. During high-competition periods (e.g., Q4 holidays, end-of-month pushes, or industry-specific peaks), inventory on major platforms like Google and Meta becomes highly constrained. The algorithm aggressively increases bids to win auctions against competitors, rapidly burning through capital for the same (or lower) conversion volume.



**The Diagnostic Conclusion:**
The data illustrates a clear point of **Diminishing Marginal Returns**: at these spend levels, the enterprise is over-saturating the available audience pool. Every additional dollar spent during these seasonal spikes is actually destroying global efficiency rather than capturing incremental value.

**Strategic Pivot (Optimizing for Efficiency):**
Rather than engaging in unprofitable bidding wars during seasonal spikes, the enterprise must adopt an agile diversification strategy:

1. **Channel Rotation:** Shift top-of-funnel acquisition budgets away from saturated platforms (Google/Meta) toward lower-CPM emerging channels (e.g., TikTok Video) when CPC thresholds are breached.
2. **Funnel Shift:** During hyper-competitive weeks, pause cold-audience acquisition and reallocate that budget entirely to high-ROAS retention channels (Email/SMS) and warm-audience retargeting.

## 8. The "Surgical" 3-Tier Strategy & Reallocation Guardrails
Rather than using broad-stroke adjustments, we apply a tiered reduction model based on historical loss probability to safely extract wasted spend.

### Phase 1: Capital Extraction (The Risk Tiers)
| TIER | CPA RANGE | RISK PROFILE | STRATEGIC ACTION | LOGIC |
| :--- | :--- | :--- | :--- | :--- |
| **1. Efficiency Trap** | $100 - $199 | Low/Medium | **Reduce Budget (40%)** | Profitable, but 3x less efficient than Star Segments. |
| **2. Danger Zone** | $200 - $249 | High | **Aggressive Cut (80%)** | High statistical probability of capital erosion. |
| **3. Death Zone** | $250+ | Absolute | **Hard Pause (Kill)** | Zero mathematical path to profitability in this segment. |

### Phase 2: Capital Injection (Industry Standard Guardrails)
Once the inefficient capital is "freed," we cannot simply dump the entire sum into our winning campaigns. To model hyper-realistic media buying operations, our Reallocation Engine applies two strict industry-standard guardrails to the injection phase:

1. **The 20% Budget Scale Cap (Protecting the "Learning Phase"):** On major platforms (Meta/Google), scaling a daily budget by more than 20% triggers a "Significant Edit." This forces the algorithm back into its learning phase, temporarily destroying performance. Our engine strictly caps all daily budget injections at 20% to maintain algorithmic momentum.
   
2. **The 80% Marginal Efficiency Penalty (Diminishing Returns):**
   When forcing an algorithm to spend more money, it must bid on less-qualified audiences. To account for this realistic decay in performance, the simulation applies a strict 20% efficiency penalty to all newly injected capital, ensuring our revenue projections are safe and mathematically sound.

In [ ]:
## --- CONFIGURATION: FORCE FULL DISPLAY ---
# This ensures Google Colab does not hide columns, truncate text, or limit rows
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None) # Force Colab to print ALL rows

# --- EXECUTING THE 3-TIER SURGICAL STRATEGY ---
def calculate_tier_action(cpa):
    """
    Applies the Surgical 3-Tier Strategy based on dynamic CPA thresholds.
    Reads the 'cpa' column and returns:
    1. Tier Label (Risk Category)
    2. Action (Strategic Command)
    3. Adjustment (Budget Multiplier)
    """
    if 100 <= cpa < 200:
        action, adjustment, tier = "REDUCE_BUDGET", -0.40, "Tier 1: Efficiency Trap"
    elif 200 <= cpa < 250:
        action, adjustment, tier = "AGGRESSIVE_CUT", -0.80, "Tier 2: Danger Zone"
    elif cpa >= 250:
        action, adjustment, tier = "HARD_PAUSE", -1.0, "Tier 3: Death Zone"
    else:
        action, adjustment, tier = "MAINTAIN/SCALE", 0.0, "Star Segment"

    return pd.Series([tier, action, adjustment])

# Apply the logic to the dataframe
df[['Tier', 'Action', 'Adj']] = df['cpa'].apply(calculate_tier_action)

# --- VIEWING THE RESULTS ---
# Define exactly what we want to see (The "Before" and "After" context)
audit_view = ['date', 'platform', 'ad_spend', 'cpa', 'Tier', 'Action', 'Adj']

print("--- STRATEGIC ACTION PLAN (FULL VISIBILITY) ---")
print("Complete audit of all operational days sorted by CPA risk:")

# Removed .head(10) so it prints the entire dataset
display(df[audit_view].sort_values(by='cpa', ascending=False))

## 10. The Reallocation Strategy: Where Does the Money Go?
**"Cost reduction without reallocation is just shrinking the business."** To achieve our new **5.21X ROAS frontier**, the **\$820,366** in liquidity rescued from our inefficient tiers (Efficiency Trap, Danger Zone, and Death Zone) must be surgically injected into validated, high-capacity "Star Segments."

**The Logic (The "Why"):**
We do not guess where to put the money. We mathematically isolate the daily combinations of `Platform` + `Campaign Type` that consistently yield a CPA under **\$100** and a high ROAS. These are our "Capital Destinations."

By routing our recovered budget exclusively into these specific funnels using strict 20/80 guardrails, we acquire net-new conversion volume at a massive discount compared to the enterprise average. This strategy ultimately drives **\$1.2M in True Net-New Revenue**, while simultaneously retaining **\$463,264** in unspent capital to protect bottom-line margins.

In [ ]:
# ==============================================================================
# 10. ENHANCED DAILY REALLOCATION ENGINE (SOURCE -> DESTINATION DETAIL)
# ==============================================================================
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Calculate Freed Capital, Lost Revenue, & Source Metrics
df['freed_capital'] = np.where(df['Adj'] < 0, df['ad_spend'] * abs(df['Adj']), 0)
df['lost_revenue'] = np.where(df['Adj'] < 0, df['revenue'] * abs(df['Adj']), 0)

reallocation_results = []

# 2. Simulate the Reallocation day-by-day
for date, group in df.groupby('date'):
    total_freed_today = group['freed_capital'].sum()

    # Identify the specific campaigns we are CUTTING today
    cuts = group[group['freed_capital'] > 0].copy()

    if total_freed_today == 0:
        continue

    # ANOVA Gate: Only Meta and TikTok can receive injections
    stars = group[
        (group['cpa'] < 100) &
        (group['cpa'] > 0) &
        (group['ad_spend'] > 0) &
        (group['platform'].isin(['TikTok Ads', 'Meta Ads']))
    ]

    if stars.empty:
        for _, cut in cuts.iterrows():
            reallocation_results.append({
                'date': date,
                'source_platform': cut['platform'],
                'source_cpa': cut['cpa'],
                'freed_from_source': cut['freed_capital'],
                'total_freed_today': total_freed_today,
                'dest_platform': 'None (Pure Savings)',
                'pre_injection_spend': 0,
                'actual_injected_capital': 0,
                'unspent_savings': cut['freed_capital'],
                'added_revenue': 0
            })
        continue

    total_star_roas = stars['roas'].sum()

    # OPTIMIZATION: Calculate source metrics outside the stars loop
    source_summary = ", ".join(cuts['platform'].unique())

    # FIX: Calculate true weighted CPA instead of a simple mean
    safe_conversions = cuts['conversions'].sum()
    avg_source_cpa = cuts['ad_spend'].sum() / safe_conversions if safe_conversions > 0 else 0

    for _, star in stars.iterrows():
        # Safeguard against zero-division for allocation percentage
        allocation_pct = star['roas'] / total_star_roas if total_star_roas > 0 else 0
        attempted_injection = total_freed_today * allocation_pct

        max_allowed_injection = star['ad_spend'] * 0.20
        actual_injection = min(attempted_injection, max_allowed_injection)
        unspent_savings = attempted_injection - actual_injection

        diminishing_return_factor = 0.80
        new_conv = (actual_injection * diminishing_return_factor) / star['cpa']
        aov = star['revenue'] / star['conversions'] if star['conversions'] > 0 else 0
        new_rev = new_conv * aov

        reallocation_results.append({
            'date': date,
            'source_platform': source_summary,
            'source_cpa': avg_source_cpa,
            'freed_from_source': attempted_injection, # FIX: Prevents double-counting in ledger
            'total_freed_today': total_freed_today,
            'dest_platform': star['platform'],
            'pre_injection_spend': star['ad_spend'],
            'actual_injected_capital': actual_injection,
            'unspent_savings': unspent_savings,
            'added_revenue': new_rev
        })

# 3. Macro Results & Final Audited Enterprise Impact
realloc_df = pd.DataFrame(reallocation_results)

# --- CLEAN THE DATE FORMAT TO REMOVE 00:00:00 ---
realloc_df['date'] = pd.to_datetime(realloc_df['date']).dt.strftime('%Y-%m-%d')

total_recovered = df['freed_capital'].sum()
total_lost_revenue = df['lost_revenue'].sum()

total_actual_injected = realloc_df['actual_injected_capital'].sum()
total_unspent_savings = realloc_df['unspent_savings'].sum()
total_new_rev = realloc_df['added_revenue'].sum()

total_revenue_baseline = df['revenue'].sum()
total_spend_baseline = df['ad_spend'].sum()
baseline_roas = total_revenue_baseline / total_spend_baseline if total_spend_baseline > 0 else 0

# THE FOOLPROOF MATH
final_revenue = (total_revenue_baseline - total_lost_revenue) + total_new_rev
final_spend = total_spend_baseline - total_recovered + total_actual_injected
final_roas = final_revenue / final_spend if final_spend > 0 else 0

true_net_revenue = final_revenue - total_revenue_baseline

# Calculate Profit Uplift to match Power BI
baseline_profit = total_revenue_baseline - total_spend_baseline
final_profit = final_revenue - final_spend
profit_uplift = final_profit - baseline_profit

print("=========================================================")
print(" 📈 FINAL PORTFOLIO OPTIMIZATION IMPACT (AUDITED)")
print("=========================================================")
print(f"Total Capital Recovered:      ${total_recovered:,.2f}")
print(f"Safely Reallocated (20% Cap): ${total_actual_injected:,.2f}")
print(f"Pure Cash Savings (Withheld): ${total_unspent_savings:,.2f}")
print("---------------------------------------------------------")
print(f"Gross New Revenue Generated:  ${total_new_rev:,.2f}")
print(f"Revenue Forfeited from Cuts: -${total_lost_revenue:,.2f}")
print(f"TRUE Net-New Enterprise Rev:  ${true_net_revenue:,.2f}")
print(f"PROFIT UPLIFT (Rev + Savings):${profit_uplift:,.2f}")
print("---------------------------------------------------------")
print(f"OLD Global Enterprise ROAS:   {baseline_roas:.2f}X")
print(f"NEW Global Enterprise ROAS:   {final_roas:.2f}X")
print("=========================================================\n")

# ==============================================================================
# 10. INTERACTIVE AUDIT LEDGER
# ==============================================================================
print("\n🔎 FULL CAPITAL FLOW AUDIT")

dropdown = widgets.Dropdown(
    options=['Select a Date...'] + sorted(list(realloc_df['date'].unique())),
    value='Select a Date...',
    description='Audit Date:',
)
output = widgets.Output()

def update_table(change):
    with output:
        clear_output()
        selected_date = change.new
        if selected_date != 'Select a Date...':
            # 1. Filter the data
            filtered_df = realloc_df[realloc_df['date'] == selected_date].copy()

            # --- Calculate Daily Totals as new columns ---
            filtered_df['daily_total_injected'] = filtered_df['actual_injected_capital'].sum()
            filtered_df['daily_total_unspent'] = filtered_df['unspent_savings'].sum()
            filtered_df['daily_total_added_rev'] = filtered_df['added_revenue'].sum()

            # 2. Reorder columns logically (Source -> Destination)
            col_order = [
                'date', 'source_platform', 'source_cpa', 'freed_from_source',
                'total_freed_today', 'dest_platform', 'pre_injection_spend',
                'actual_injected_capital', 'daily_total_injected',
                'unspent_savings', 'daily_total_unspent',
                'added_revenue', 'daily_total_added_rev'
            ]
            filtered_df = filtered_df[col_order]

            # 3. Apply safe formatting & styling fixes
            styled_table = filtered_df.style.format({
                'source_cpa': '${:,.2f}',
                'freed_from_source': '${:,.2f}',
                'total_freed_today': '${:,.2f}',
                'pre_injection_spend': '${:,.2f}',
                'actual_injected_capital': '${:,.2f}',
                'daily_total_injected': '${:,.2f}',
                'unspent_savings': '${:,.2f}',
                'daily_total_unspent': '${:,.2f}',
                'added_revenue': '${:,.2f}',
                'daily_total_added_rev': '${:,.2f}'
            }).background_gradient(subset=['added_revenue', 'daily_total_added_rev'], cmap='Greens') \
              .set_properties(**{'color': '#9b1c1c', 'font-weight': 'bold'}, subset=['source_platform', 'source_cpa', 'freed_from_source']) \
              .set_properties(**{'color': '#065f46', 'font-weight': 'bold'}, subset=['dest_platform', 'actual_injected_capital']) \
              .set_properties(**{'background-color': '#f3f4f6', 'font-weight': 'bold'}, subset=['daily_total_injected', 'daily_total_unspent']) \
              .hide(axis='index')

            display(styled_table)

dropdown.observe(update_table, names='value')
display(dropdown)
display(output)

## 11. Executive Conclusion & Strategic Next Steps
By programmatically applying this 3-Tier Reallocation Engine, we protect the algorithmic learning phase while ruthlessly mitigating capital risk.

**Immediate Action Items:**
1. **Deploy Automated Guardrails (The Kill-Switch):** Implement automated platform rules to immediately pause any campaign exceeding the \$250 CPA threshold (Death Zone) to prevent daily capital bleed.
2. **Execute Capital Reallocation:** Reroute the \$820k identified in the inefficient tiers toward validated Star Segments, strictly adhering to the 20% daily scale cap to protect algorithmic momentum.
3. **Continuous Monitoring:** Implement a daily automated script using the variance thresholds defined in this audit to catch CPC and CPA inflation before it erodes net profit.

**Note on Financial Rigor:** Unlike basic analytical models that assume unrealistic linear scaling, this simulation was built under strict enterprise constraints. It explicitly accounts for **Diminishing Marginal Returns** (applying a 20% efficiency penalty to all new budget injections) and **Revenue Forfeiture** (subtracting the lost sales from paused campaigns). Even under these highly conservative financial guardrails, the strategy successfully drives a **5.21X global enterprise ROAS**, mathematically proving that the reallocation thesis holds true under real-world operational stress.

## 💡 Conclusions & Limitations
### The Strategic Conclusion
This engine successfully proves that even highly profitable macro-environments contain significant micro-level waste. By shifting the analytical focus from top-of-funnel engagement (CTR/Clicks) to bottom-of-funnel risk thresholds (CPA), the business can proactively cut funding to statistically losing days. The simulation proves that applying these strict financial guardrails can theoretically capture **$1.2M in optimized profit** and push overall efficiency to a **5.21X ROAS**.

### Model Limitations & Real-World Considerations
To transition this theoretical simulation into a live production environment, several real-world marketing mechanics must be accounted for:
1. **The Attribution Delay:** The current daily simulation evaluates same-day ad spend against same-day conversions. In reality, a user might click an ad on Tuesday but convert on Friday. The model currently does not account for multi-day attribution windows.
2. **Algorithmic Reset Risk:** Ad platforms (Meta, Google) utilize machine learning bidding algorithms. Constantly reallocating budget on a daily, surgical basis can disrupt the platform's "learning phase," potentially suppressing overall campaign performance.
3. **Audience Saturation (Diminishing Returns):** The engine assumes that capital moved into a high-yield campaign will convert at the exact same historical efficiency. In reality, injecting sudden capital into a campaign often leads to audience saturation and an eventual spike in CPA.

In [ ]:
import pandas as pd
from google.colab import files

print("⏳ Initiating Final Export Protocol...\n")

# ---------------------------------------------------------
# 1. Export the ANOVA Results Table
# ---------------------------------------------------------
anova_clean.to_csv('anova_results.csv', index=False)
files.download('anova_results.csv')

# ---------------------------------------------------------
# 2. Export the 136 Losing Days Table
# ---------------------------------------------------------
losing_days.to_csv('136_losing_days.csv', index=False)
files.download('136_losing_days.csv')

# ---------------------------------------------------------
# 3. Export Metric Variance: Baseline vs. Anomaly
# ---------------------------------------------------------
# We MUST use index=True here because metrics ('ctr', 'cpc', etc.) are the index
variance_profile.to_csv('metric_variance.csv', index=True)
files.download('metric_variance.csv')

# ---------------------------------------------------------
# 4. Export Probability Audit: CPA Threshold Sensitivity
# ---------------------------------------------------------
threshold_df.to_csv('cpa_threshold_sensitivity.csv', index=False)
files.download('cpa_threshold_sensitivity.csv')

# ---------------------------------------------------------
# 5. Export the Complete 3-Tier Surgical Strategy Action Plan
# ---------------------------------------------------------
# Lock in the exact view (filtered columns + sorted by CPA risk)
surgical_strategy_df = df[audit_view].sort_values(by='cpa', ascending=False)
surgical_strategy_df.to_csv('surgical_strategy_full_audit.csv', index=False)
files.download('surgical_strategy_full_audit.csv')

# ---------------------------------------------------------
# 6. Export the Complete Reallocation Engine Ledger
# ---------------------------------------------------------
# index=False to keep the data perfectly clean for Power BI
realloc_df.to_csv('daily_reallocation_ledger.csv', index=False)
files.download('daily_reallocation_ledger.csv')

from google.colab import files

# 1. Update the view to include the missing revenue columns
audit_view_updated = ['date', 'platform', 'ad_spend', 'revenue', 'cpa', 'Tier', 'Action', 'Adj', 'lost_revenue']

# 2. Lock in the new view
surgical_strategy_df = df[audit_view_updated].sort_values(by='cpa', ascending=False)

# 3. Export and download the updated file
surgical_strategy_df.to_csv('surgical_strategy_full_audit.csv', index=False)
files.download('surgical_strategy_full_audit.csv')

from google.colab import files

# ---------------------------------------------------------
# Export the Raw Ingested Performance Metrics
# ---------------------------------------------------------
# We use the variable 'df' because that is where the SQL query saved the data
df.to_csv('vw_performance_metrics.csv', index=False)

# Trigger the browser download
files.download('vw_performance_metrics.csv')

print("✅ vw_performance_metrics exported successfully!")

print("✅ All 6 core analytical files have been successfully exported for BI integration!")